# OpenPlaque RCA 3-D Aorta-Anchored Gallery v3

This replaces the failed per-slice pulmonary/aorta heuristic. It identifies the **3-D aortic contrast component through the arch**, then searches only around the true ascending-aortic root for proximal RCA candidates. No sliders, widgets, clicking, or manual coordinates.

Use **Runtime → Run all**. Google Drive mounts first.

In [ ]:
# ALWAYS FIRST
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch rca-centerline-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas

import os, sys, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi

SRC=Path('/content/OpenPlaque/src')
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))
from openplaque.study import OpenPlaqueStudy
from openplaque.rca_aorta3d import segment_aorta_3d, find_rca_candidates_from_aorta3d
print('Dependencies ready.')

In [ ]:
ROOT=Path('/content/drive/MyDrive/OpenPlaque')
DRIVE_ZIP=ROOT/'Full_DICOM.zip'
LOCAL_ZIP=Path('/content/Full_DICOM.zip')
EXTRACT_ROOT='/content/full_dicom_aorta3d_v3'
if not DRIVE_ZIP.exists(): raise FileNotFoundError(DRIVE_ZIP)
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    print('Copying Full_DICOM.zip locally...',flush=True); shutil.copyfile(DRIVE_ZIP,LOCAL_ZIP)
shutil.rmtree(EXTRACT_ROOT,ignore_errors=True)
print('Extracting/scanning...',flush=True); t=time.time()
study=OpenPlaqueStudy(str(LOCAL_ZIP),extract_root=EXTRACT_ROOT)
source_img,source,source_files=study.load_series(7)
print(f'Loaded series 7 in {time.time()-t:.1f}s; shape={source.shape}; spacing={source_img.GetSpacing()}')

In [ ]:
print('Finding 3-D aortic component...',flush=True); t=time.time()
aorta=segment_aorta_3d(source,source_img,threshold_hu=340.0,work_spacing_mm=1.0)
print(f'Aorta segmentation finished in {time.time()-t:.1f}s')
print('Ascending-aorta tracked slices:',len(aorta.ascending_centers_zyx))
print('Root-search slice range:',int(aorta.root_slice_indices.min()),'..',int(aorta.root_slice_indices.max()))
print('Finding RCA candidates around true aortic component...',flush=True); t=time.time()
cands=find_rca_candidates_from_aorta3d(source,source_img,aorta,n=8)
print(f'Candidate search finished in {time.time()-t:.1f}s; {len(cands)} candidates')
df=pd.DataFrame([dict(candidate=f'R{i+1}',z=c.z,y=c.y,x=c.x,score=c.score,HU=c.hu,vesselness=c.vesselness,radius_mm=c.local_radius_mm,distance_from_aorta_mm=c.distance_from_aorta_mm,support=c.support_slices) for i,c in enumerate(cands)])
display(df)

In [ ]:
OUTDIR=ROOT/'RCA_Aorta3D_v3'; OUTDIR.mkdir(parents=True,exist_ok=True)
center_by_z={int(round(z)):(float(y),float(x)) for z,y,x in aorta.ascending_centers_zyx}
valid_z=sorted(center_by_z)
# Focus validation on the DICOM-inferior root-search portion, not the whole arch.
zs=np.linspace(int(aorta.root_slice_indices.min()),int(aorta.root_slice_indices.max()),12).round().astype(int)
fig,axes=plt.subplots(3,4,figsize=(16,12))
for ax,z in zip(axes.ravel(),zs):
    ax.imshow(source[z],cmap='gray',vmin=-200,vmax=800)
    sl=aorta.mask[z]
    # Show the boundary of the 3-D aortic component.
    edge=sl ^ ndi.binary_erosion(sl)
    yy,xx=np.where(edge)
    ax.scatter(xx,yy,s=1)
    if z in center_by_z:
        cy,cx=center_by_z[z]; ax.scatter([cx],[cy],s=80,facecolors='none'); ax.text(cx+5,cy,'ascending aorta')
    ax.set_title(f'3-D aorta validation z={z}'); ax.axis('off')
plt.tight_layout(); p=OUTDIR/'RCA_3D_aorta_anchor_validation.png'; fig.savefig(p,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig); print('Saved',p)

In [ ]:
fig,axes=plt.subplots(4,2,figsize=(12,20))
for i,ax in enumerate(axes.ravel()):
    if i>=len(cands): ax.axis('off'); continue
    c=cands[i]; z,y,x=c.z,c.y,c.x; r=90
    y0,y1=max(0,y-r),min(source.shape[1],y+r+1); x0,x1=max(0,x-r),min(source.shape[2],x+r+1)
    ax.imshow(source[z,y0:y1,x0:x1],cmap='gray',vmin=-200,vmax=800)
    ax.scatter([x-x0],[y-y0],s=90,facecolors='none')
    ax.set_title(f'R{i+1}: z={z} HU={c.hu:.0f} support={c.support_slices} dA={c.distance_from_aorta_mm:.1f}mm')
    ax.axis('off')
plt.tight_layout(); p=OUTDIR/'RCA_3D_candidate_closeups.png'; fig.savefig(p,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig); print('Saved',p)

In [ ]:
if cands:
    c=cands[0]; dz=max(1,int(round(1.2/source_img.GetSpacing()[2])))
    seq=np.arange(max(0,c.z-5*dz),min(source.shape[0],c.z+5*dz+1),dz,dtype=int)
    fig,axes=plt.subplots(3,4,figsize=(16,12))
    for ax in axes.ravel(): ax.axis('off')
    for ax,z in zip(axes.ravel(),seq):
        r=75; y,x=c.y,c.x; y0,y1=max(0,y-r),min(source.shape[1],y+r+1); x0,x1=max(0,x-r),min(source.shape[2],x+r+1)
        ax.imshow(source[z,y0:y1,x0:x1],cmap='gray',vmin=-200,vmax=800); ax.set_title(f'z={z} ({(z-c.z)*source_img.GetSpacing()[2]:+.1f} mm from R1)'); ax.axis('off')
    plt.tight_layout(); p=OUTDIR/'RCA_R1_dense_sequence_3Danchor.png'; fig.savefig(p,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig); print('Saved',p)
print('DONE. Upload the aorta-anchor validation, candidate closeups, and R1 dense sequence.')